In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Step 2: Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import os

# Define model_name and new_model
model_name = "NousResearch/Llama-2-7b-chat-hf"
adapter_weights_dir = "/content/drive/MyDrive/Projects/Deep Learning for NLP/Finance_Chatbot"  # Change this to your adapter weights path

# Step 3: Clear GPU memory
torch.cuda.empty_cache()

# Step 4: Load the base model in FP16
try:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        low_cpu_mem_usage=True,
        return_dict=True,
        torch_dtype=torch.float16,
        device_map="auto",  # Automatically splits the model across available GPUs
    )

    # Step 5: Load the adapter model from Google Drive
    model = PeftModel.from_pretrained(base_model, adapter_weights_dir)

    # Step 6: Merge the model and unload unnecessary weights
    model = model.merge_and_unload()

    # Step 7: Reload the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Step 8: Save the model and tokenizer to a specified location
    output_dir = "/content/drive/MyDrive/Projects/Deep Learning for NLP"  # Change this to your desired output path
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Step 9: List the contents to ensure files are saved
    print("Contents of output directory:", os.listdir(output_dir))

except RuntimeError as e:
    if "out of memory" in str(e):
        print("Out of memory error. Try using a smaller model or increasing GPU memory.")
        torch.cuda.empty_cache()
    else:
        raise e
except ValueError as e:
    print(f"ValueError: {e}")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

Contents of output directory: ['combined_Text.txt', 'Finance_Chatbot', 'config.json', 'generation_config.json', 'model-00001-of-00003.safetensors', 'model-00002-of-00003.safetensors', 'model-00003-of-00003.safetensors', 'model.safetensors.index.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'tokenizer.model', 'tokenizer.json']


In [3]:
!pip install huggingface_hub

In [27]:
def get_token_from_drive(file_path):
    """Read the token from a text file in Google Drive."""
    with open(file_path, 'r') as file:
        token = file.read().strip()
    return token

# Load the ngrok auth token and Hugging Face token from Google Drive text files
ngrok_token_path = '/content/drive/MyDrive/Projects/Deep Learning for NLP/ngrok_token.txt'  # Update with your file path
huggingface_token_path = '/content/drive/MyDrive/Projects/Deep Learning for NLP/Huggingface_token.txt'  # Update with your file path

ngrok_auth_token = get_token_from_drive(ngrok_token_path)
huggingface_token = get_token_from_drive(huggingface_token_path)


In [29]:
from huggingface_hub import login

# Login
login(huggingface_token)


In [7]:
from huggingface_hub import create_repo

# Replace with your desired model repo name
repo_name = "Satyamk122/Satyam_FinanceGPT_Finetunned"  # Change this to your desired model repository name
create_repo(repo_name, exist_ok=True)


RepoUrl('https://huggingface.co/Satyamk122/Satyam_FinanceGPT_Finetunned', endpoint='https://huggingface.co', repo_type='model', repo_id='Satyamk122/Satyam_FinanceGPT_Finetunned')

In [9]:
import os

# Set your Git username and email
os.system('git config --global user.name "Satyam Kulkarni"')  # Replace with your name
os.system('git config --global user.email "satyamkulkarni122@gmail.com"')  # Replace with your email


0

In [16]:
import os
from huggingface_hub import login, upload_file

# Define your model directory
model_dir = r"/content/drive/MyDrive/Projects/Deep Learning for NLP"

# List of files to upload
files_to_upload = [
    "model-00002-of-00003.safetensors",
    "added_tokens.json",
    "config.json",
    "generation_config.json",
    "model.safetensors.index.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer.model",
    "tokenizer_config.json",
    "model-00001-of-00003.safetensors",
    "model-00003-of-00003.safetensors"
]

# Iterate through the files and upload each one
for filename in files_to_upload:
    file_path = os.path.join(model_dir, filename)
    upload_file(
        path_or_fileobj=file_path,
        path_in_repo=filename,
        repo_id="Satyamk122/Satyam_FinanceGPT_Finetunned",
        repo_type="model"  # Specify the type of the repository (model, dataset, etc.)
    )
    print(f"Uploaded {filename} to Hugging Face.")

print("All files uploaded successfully.")


model-00002-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Uploaded model-00002-of-00003.safetensors to Hugging Face.
Uploaded added_tokens.json to Hugging Face.
Uploaded config.json to Hugging Face.
Uploaded generation_config.json to Hugging Face.
Uploaded model.safetensors.index.json to Hugging Face.
Uploaded special_tokens_map.json to Hugging Face.
Uploaded tokenizer.json to Hugging Face.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Uploaded tokenizer.model to Hugging Face.
Uploaded tokenizer_config.json to Hugging Face.


model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Uploaded model-00001-of-00003.safetensors to Hugging Face.


model-00003-of-00003.safetensors:   0%|          | 0.00/3.59G [00:00<?, ?B/s]

Uploaded model-00003-of-00003.safetensors to Hugging Face.
All files uploaded successfully.


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

def load_model(model_name):
    # Load the pre-trained model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,  # Using FP16 for memory efficiency
        device_map='auto'  # Automatically assigns the model to available GPUs
    )

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token  # Align padding
    tokenizer.padding_side = "right"

    return model, tokenizer

# Load the model and tokenizer
model_name = "Satyamk122/Satyam_FinanceGPT_Finetunned"
model, tokenizer = load_model(model_name)

def perform_inference(model, tokenizer, input_text):
    # Create a text generation pipeline
    pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=100)

    # Format the prompt with special tokens
    prompt = f"<s>[INST] {input_text} [/INST]"

    # Generate output
    result = pipe(prompt)

    # Extract the generated text from the result
    output_text = result[0]['generated_text']

    return output_text


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

In [14]:
# Example usage
input_text = "What is meaning of bond market?"
output_text = perform_inference(model, tokenizer, input_text)
print("Output:", output_text)

Output: <s>[INST] What is meaning of bond market? [/INST] The bond market is the market for bonds, which are debt securities. It is where investors can buy and sell bonds. The market is made up of various types of investors, including individuals, companies, and governments. The bond market is a way for companies and governments to borrow money from investors to finance their operations. The investors receive regular interest payments and the
